# 취약점 탐지 에이전트

보안 팀은 공격자보다 먼저 메모리 안전성 버그를 찾아내고 싶어 하지만, 기존 도구로는 쉽지 않습니다. 정적 분석기는 거짓 양성을 너무 많이 쏟아내 검토자가 읽기를 포기하게 만들고, 퍼저는 무언가를 찾아내기 전에 진입점마다 손으로 하니스를 작성해야 합니다. 이 쿡북에서는 **Claude Agent SDK**로 취약점 탐색 에이전트를 만드는 방법을 보여 줍니다. 이 에이전트는 Claude Code의 내장 `Read`, `Grep`, `Glob` 도구로 소스 코드를 읽고, 어떤 입력이 메모리를 손상시킬 수 있는지 추론하며, 검토자가 곧바로 조치할 수 있는 형태로 발견 사항을 작성합니다.

**이 쿡북을 마치면 다음을 할 수 있습니다.**

- 부트스트랩 후 인터뷰 방식의 위협 모델링을 멀티턴 `ClaudeSDKClient` 세션으로 실행해 `THREAT_MODEL.md` 작성하기
- 직접 만든 파일 접근 도구 대신 내장 `Read`/`Grep`/`Glob` 도구로 에이전트 탐색 루프 구동하기
- 탐색, 선별, 보고를 각각 별개의 `query()` 호출로 이어 붙여 스키마를 준수하는 JSON 생성하기

## 사전 준비

**필요한 사전 지식:**
- `async`/`await`을 포함한 Python 기초
- 45줄짜리 파일을 읽고 `memcpy`를 알아볼 정도의 C 지식

**필요한 도구:**
- Python 3.11 이상
- Node.js 18 이상과 Claude Code CLI: `npm install -g @anthropic-ai/claude-code`
- Anthropic API 키 ([여기서 발급](https://console.anthropic.com))

**실제 대상에 사용할 때 필수:** 분석하려는 코드를 평가할 권한이 있어야 합니다. 이 노트북에는 버그가 심어진 작고 자체 완결된 `canary.c`가 포함되어 있어, 프로덕션 코드를 건드리지 않고도 전체 과정을 실행해 볼 수 있습니다.

## 1단계: 환경과 평가 맥락 설정하기

모든 에이전트에 `system_prompt`로 전달할 `ENGAGEMENT_CONTEXT` 블록을 정의합니다. 이번 평가의 범위(코드 소유자의 승인, 격리된 읽기 전용 샌드박스, 책임 있는 공개를 향하는 발견 사항)를 기록해 두어, 파이프라인의 모든 단계가 동일하게 문서화된 기본 규칙 아래에서 동작하게 합니다. 실제 대상에 사용할 때는 이 세 가지 전제를 반드시 참으로 유지하세요.

> **사이버 안전장치에 대한 참고:** Claude는 API 계층에서 [실시간 사이버 안전장치](https://support.claude.com/en/articles/14604842-real-time-cyber-safeguards-on-claude)를 적용합니다. 실제 코드베이스를 다루다가 이 안전장치가 작동한다면, 해당 페이지에서 사이버 검증 프로그램(CVP)에 신청하세요. 전문가가 정당한 이중 용도 보안 작업을 최소한의 중단으로 계속할 수 있게 해 주는 무료 신청 기반 프로그램입니다.

In [1]:
%%capture
%pip install -U claude-agent-sdk python-dotenv

In [2]:
import json
from collections.abc import AsyncIterator
from pathlib import Path

from dotenv import load_dotenv

from claude_agent_sdk import (
    AssistantMessage,
    ClaudeAgentOptions,
    ClaudeSDKClient,
    Message,
    ResultMessage,
    TextBlock,
    ToolUseBlock,
    query,
)

load_dotenv()

MODEL_NAME = "claude-opus-4-7"
# This notebook expects to be run from the claude_agent_sdk/ directory
# (Jupyter's default when you open the file from there). The assert makes
# the failure explicit if the kernel was started elsewhere.
TARGET_DIR = Path("vulnerability_detection_agent/canary").resolve()
assert TARGET_DIR.is_dir(), f"run this notebook from claude_agent_sdk/ (got cwd={Path.cwd()})"

ENGAGEMENT_CONTEXT = """\
## Engagement context

This is authorized security research conducted as a defensive security
assessment on a self-contained canary target vendored in this notebook. The
target is read-only source (no execution). Findings are collected for
demonstration and responsible-disclosure workflow testing.
"""


async def collect(stream: AsyncIterator[Message]) -> str:
    """Consume an Agent SDK message stream; print tool calls; return final text.

    Both ``query()`` and ``ClaudeSDKClient.receive_response()`` return an
    ``AsyncIterator[Message]`` that terminates after a ``ResultMessage``.
    This is the same ``async for msg in ...`` loop the other notebooks in this
    series write inline; it is factored out here because this notebook runs
    the loop four times (TM bootstrap, TM interview, find, triage) and the
    ``isinstance`` ladder would otherwise repeat verbatim.
    """
    final = ""
    async for msg in stream:
        if isinstance(msg, AssistantMessage):
            for block in msg.content:
                if isinstance(block, ToolUseBlock):
                    args = str(block.input)
                    args = args if len(args) <= 120 else args[:120] + "...}"
                    print(f"  [tool] {block.name} {args}")
                elif isinstance(block, TextBlock) and block.text.strip():
                    final += block.text
        elif isinstance(msg, ResultMessage) and msg.is_error:
            raise RuntimeError(msg.result)
    return final


print(f"Model: {MODEL_NAME}")

Model: claude-opus-4-7


## 2단계: 카나리 대상 불러오기

`vulnerability_detection_agent/canary/canary.c`는 메모리 안전성 버그 세 개(힙 버퍼 오버플로, 스택 버퍼 오버플로, 사용 후 해제)를 일부러 심어 둔 45줄 남짓의 C 프로그램이며, 각각 입력 앞부분의 서로 다른 "매직 바이트"로 도달할 수 있습니다. 버그에는 표시가 없으므로, 4단계의 탐색 에이전트는 실제 코드에서 그러하듯 로직을 읽어 스스로 찾아내야 합니다. 여러분의 코드로 시험해 볼 준비가 되면 `TARGET_DIR`을 여러분의 체크아웃으로 가리키게 하세요.

In [3]:
print((TARGET_DIR / "canary.c").read_text())

// canary.c
// Entry: ./canary <input_file>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

static void parse_alpha(const unsigned char *data, size_t len) {
    unsigned char *buf = malloc(32);
    memcpy(buf, data, len);
    printf("alpha: %02x\n", buf[0]);
    free(buf);
}

static void parse_bravo(const unsigned char *data, size_t len) {
    char name[16];
    memcpy(name, data, len);
    name[15] = 0;
    printf("bravo: %s\n", name);
}

static void parse_charlie(const unsigned char *data, size_t len) {
    char *p = malloc(64);
    if (len > 0 && data[0] == 0xff) {
        free(p);
    }
    memcpy(p, data, len < 64 ? len : 64);
    printf("charlie: %p\n", (void *)p);
}

int main(int argc, char **argv) {
    if (argc < 2) return 1;
    FILE *f = fopen(argv[1], "rb");
    if (!f) return 1;
    unsigned char buf[4096];
    size_t n = fread(buf, 1, sizeof buf, f);
    fclose(f);
    if (n < 1) return 1;
    switch (buf[0]) {
        case 'A': parse_alpha(buf + 1, n - 1); br

## 3단계: 대상 위협 모델링하기 (부트스트랩 후 인터뷰)

위협 모델은 특정 버그와 무관하게 "이 시스템에서 무엇이 잘못될 수 있고, 누가 그렇게 만들 것이며, 어떤 결과가 중요한가"에 답합니다. 위협("공격자가 신뢰할 수 없는 파일 파싱을 통해 메모리 손상을 일으킨다")은 패치 이후에도 남지만, 취약점("31번 줄이 `len`의 범위를 검사하지 않는다")은 그렇지 않습니다. 탐색 루프는 취약점을 사냥하고, 위협 모델은 어디를 사냥할지 알려 주며 선별 단계에는 어떻게 점수를 매길지 알려 줍니다.

하나의 `ClaudeSDKClient` 세션에서 두 턴에 걸쳐 만듭니다.

1. **부트스트랩.** Claude가 내장 `Read` 도구로 코드를 읽고 모델 초안을 작성합니다(맥락, 자산, 진입점과 신뢰 경계, 위협, 그리고 코드만으로는 답할 수 없는 **미해결 질문**).
2. **인터뷰.** 애플리케이션 소유자가 미해결 질문에 답하면 Claude가 발생 가능성과 영향을 다듬은 뒤 대상 옆에 `THREAT_MODEL.md`를 작성합니다.

두 턴을 한 클라이언트 세션에 두면 인터뷰 턴이 소스를 다시 보내지 않고도 부트스트랩의 도구 결과를 볼 수 있습니다. 45줄짜리 카나리에서는 두 턴 모두 얄팍하지만, 여기서 중요한 것은 **출력의 형태**입니다. 진입점 표(실제 저장소라면 병렬 탐색 에이전트들에 나눠 줄 대상)와 미해결 질문 목록(부트스트랩에서 인터뷰로 넘기는 인계 지점)이 그것입니다.

In [4]:
(TARGET_DIR / "THREAT_MODEL.md").unlink(missing_ok=True)

TM_SCHEMA = """\
# Threat Model: <system name>
## 1. System context
## 2. Assets
| asset | description | sensitivity |
## 3. Entry points & trust boundaries
| entry_point | description | trust_boundary | reachable_assets |
## 4. Threats
| id | threat | surface | asset | impact | likelihood |
## 5. Open questions
- (things the code alone cannot answer: deployment context, which inputs are
  attacker-controlled in practice, blast radius)
"""

BOOTSTRAP_PROMPT = f"""\
You are bootstrapping a threat model from source code alone; no application
owner is available yet. Read `canary.c` in this directory and emit a draft
threat model in the schema below. Be explicit in section 5 about what you could
NOT determine from the code: those open questions are the agenda for the owner
interview. Do not write any files yet.

## Schema

{TM_SCHEMA}
"""

OWNER_ANSWERS = """\
- Deployment: `canary` is a local CLI that reads a file path from argv; it is
  not network-facing.
- Attacker control: the input file is fully attacker-controlled (think email
  attachment or downloaded file opened by the user).
- Blast radius: the process runs as the invoking user with no sandboxing;
  memory corruption is code execution as that user.
"""

INTERVIEW_PROMPT = f"""\
The application owner has now answered your open questions:

{OWNER_ANSWERS}

Refine the threat model: update likelihood and impact in section 4 using the
owner's answers, resolve every item in section 5 that the answers cover, and
add any new threats the deployment context implies. Keep the same schema, then
write the refined model to `THREAT_MODEL.md` in this directory.
"""

tm_options = ClaudeAgentOptions(
    model=MODEL_NAME,
    cwd=str(TARGET_DIR),
    system_prompt={"type": "preset", "preset": "claude_code", "append": ENGAGEMENT_CONTEXT},
    allowed_tools=["Read", "Write", "Edit"],
    disallowed_tools=["Bash"],
    permission_mode="acceptEdits",
)

async with ClaudeSDKClient(options=tm_options) as tm_agent:
    # collect() fully drains receive_response() through its terminating
    # ResultMessage, so the second query() sees a clean stream.
    await tm_agent.query(BOOTSTRAP_PROMPT)
    draft_tm = await collect(tm_agent.receive_response())
    print("--- bootstrap draft ---\n" + draft_tm + "\n")

    await tm_agent.query(INTERVIEW_PROMPT)
    await collect(tm_agent.receive_response())

tm_path = TARGET_DIR / "THREAT_MODEL.md"
if not tm_path.exists():
    raise RuntimeError("interview agent did not write THREAT_MODEL.md; check the trace above")
threat_model = tm_path.read_text()
print("--- refined THREAT_MODEL.md ---\n" + threat_model)

  [tool] Read {'file_path': 'vulnerability_detection_agent/canary/cana...}
--- bootstrap draft ---
`★ Insight ─────────────────────────────────────`
- The file dispatches on `buf[0]` into three parsers, each with a distinct memory-safety bug class: heap overflow, stack overflow, and use-after-free. This is a canonical "one bug per parser" canary.
- The `len` passed to each parser is `n - 1` (up to 4095), but buffers are sized 32/16/64, so every path is reachable with attacker-controlled overflow length from a single input file.
- Threat modeling from source alone can enumerate the *bug classes* and *entry points*, but it cannot tell you who supplies `argv[1]` in production — that determines whether these are local footguns or remote RCE primitives.
`─────────────────────────────────────────────────`

# Threat Model: canary (file-format parser CLI)

## 1. System context
A small C command-line utility invoked as `./canary <input_file>`. It opens the supplied path, reads up to 4096 bytes,

## 4단계: 에이전트 탐색 루프 실행하기

원시 Messages API였다면 이 단계는 커스텀 파일 도구와 함께 직접 작성한 `while stop_reason == "tool_use"` 루프였을 것입니다. Agent SDK는 그 모든 것을 대신 처리합니다. `allowed_tools=["Read", "Grep", "Glob"]`과 `disallowed_tools=["Bash"]`로 `query()`를 한 번 호출하면 Claude Code가 탐색-읽기-추론 루프를 알아서 돕니다. `cwd=str(TARGET_DIR)`가 에이전트를 카나리로 가리키고, `system_prompt={"type": "preset", "preset": "claude_code", "append": ...}`가 Claude Code의 기본 시스템 프롬프트(이미 에이전트에 작업 디렉터리를 알려 줍니다)를 유지하면서 우리의 평가 맥락을 덧붙이므로, 에이전트가 자기 위치를 추측할 일이 없습니다. `Bash`/`Write`/`Edit`을 주지 않았으므로 에이전트는 읽기 전용으로 남습니다.

**프롬프트에서 가장 중요한 부분은 여전히 품질 등급 루브릭입니다.** 이것이 없으면 LLM 취약점 탐색기는 찾을 수 있는 모든 널 포인터 역참조와 실패한 단언을 보고합니다. 실제 크래시이긴 하지만 거의 익스플로잇이 불가능한 것들이죠. 루브릭은 어떤 크래시 부류를 제출해야 하는지(힙/스택 오버플로, 사용 후 해제, 제어 가능한 주소 쓰기), 그리고 어떤 것이 계속 읽어 나가라는 이정표에 불과한지 알려 줍니다. 이 블록 하나가 보안 엔지니어가 조치하는 보고서와 무시하는 보고서를 가르는 차이의 대부분입니다.

프로덕션 버전이라면 `allowed_tools`에 `"Bash"`를 추가해 에이전트가 `-fsanitize=address`로 컴파일해 각 크래시를 확인하게 할 것입니다. 그것은 잠긴 컨테이너 안에서 해야 할 일이므로 이 노트북은 읽기 전용으로 남깁니다.

In [5]:
FIND_PROMPT = f"""\
Find memory-safety bugs in the target source tree using the file tools
available to you.

## Threat model

Focus on the entry points and threats identified here; you do not need to
re-derive them.

{threat_model}

## Quality tiers: what to report

**HIGH VALUE (report these):**
- heap-buffer-overflow (especially WRITE)
- heap-use-after-free / double-free
- stack-buffer-overflow
- global-buffer-overflow

**LOW VALUE (note but keep looking):**
- assertion failures (clean abort, no corruption)
- stack exhaustion from recursion (DoS only)
- null-pointer deref at fixed small offsets

## Output

For each HIGH VALUE finding emit a block:

<finding>
<id>F-NN</id>
<file>path:line</file>
<category>heap-buffer-overflow | stack-buffer-overflow | use-after-free | ...</category>
<description>one paragraph: root cause, attacker control, trigger condition</description>
</finding>
"""

find_options = ClaudeAgentOptions(
    model=MODEL_NAME,
    cwd=str(TARGET_DIR),
    system_prompt={"type": "preset", "preset": "claude_code", "append": ENGAGEMENT_CONTEXT},
    allowed_tools=["Read", "Grep", "Glob"],
    disallowed_tools=["Bash"],
)

findings_text = await collect(query(prompt=FIND_PROMPT, options=find_options))
print("\n" + findings_text)

  [tool] Glob {'pattern': '**/*'}
  [tool] Read {'file_path': 'vulnerability_detection_agent/canary/cana...}

`★ Insight ─────────────────────────────────────`
- The one-byte dispatch at line 38 makes each parser independently reachable with a trivial file prefix (`A`/`B`/`C`), so each bug is a standalone attack surface.
- `parse_bravo`'s `name[15]=0` is a common false-safety pattern: it null-terminates for the `printf`, but the `memcpy` on line 16 has already written past the 16-byte frame before that line runs.
- `parse_charlie` composes two primitives (conditional free then unconditional write) into a clean UAF whose trigger byte is attacker-chosen, which is rarer than an accidental UAF and nastier to fix.
`─────────────────────────────────────────────────`

<finding>
<id>F-01</id>
<file>canary.c:8-9</file>
<category>heap-buffer-overflow</category>
<description>`parse_alpha` allocates a fixed 32-byte heap buffer and then `memcpy`s `len` attacker-controlled bytes into it with no boun

## 5단계: 원시 발견 사항 선별하기

탐색 에이전트는 재현율에 맞춰 조정되어 있어서, 출력에는 보통 중복(두 경로로 도달한 하나의 근본 원인)이 섞이고 가끔은 성립하지 않는 발견도 들어갑니다. 선별이 그 필터입니다. 새 `query()`가 코드를 다시 읽어 각 발견을 실제 줄과 대조해 **검증하고**, 근본 원인 기준으로 **중복을 합치고**, 위협 모델의 신뢰 경계를 넘는 도달 가능성으로부터 **심각도를 다시 도출합니다**. 선별 단계가 탐색 에이전트의 심각도 점수를 물려받지 않게 일부러 막아 뒀습니다. 독립적으로 다시 도출하는 것이 과신을 잡아내는 값싼 방법이기 때문입니다.

In [6]:
TRIAGE_PROMPT = f"""\
Triage these findings against the source in this directory and the threat model
below. For each: verify it is real (cite the line), derive severity from
reachability across the trust boundaries in the threat model, and collapse
duplicates by root cause.

## Threat model

{threat_model}

## Raw findings

{findings_text}
"""

triage_options = ClaudeAgentOptions(
    model=MODEL_NAME,
    cwd=str(TARGET_DIR),
    system_prompt={"type": "preset", "preset": "claude_code", "append": ENGAGEMENT_CONTEXT},
    allowed_tools=["Read", "Grep"],
    disallowed_tools=["Bash"],
)

triaged_text = await collect(query(prompt=TRIAGE_PROMPT, options=triage_options))
print("\n" + triaged_text)

  [tool] Read {'file_path': 'vulnerability_detection_agent/canary/cana...}

`★ Insight ─────────────────────────────────────`
- All three bugs sit behind the same trust boundary (attacker-authored file → parser), reached by a single-byte dispatch in `main`. No auth, no sandbox, same-uid blast radius, so reachability is identical for all three and severity is driven by the memory-corruption primitive itself.
- F-03 is really two bugs fused at one call site: the UAF (T3) and the `printf("%p")` heap-pointer disclosure (T5). Collapsing them under F-03 is correct by root cause (both are in `parse_charlie`'s 6 lines), but the leak is what upgrades the UAF from "unreliable under ASLR" to "one-shot".
`─────────────────────────────────────────────────`

## Triage

| id | verdict | line(s) | threat | severity | notes |
| --- | --- | --- | --- | --- | --- |
| F-01 | real | canary.c:8-9 | T1 | **Critical** | `malloc(32)` then `memcpy(buf, data, len)` with `len` up to 4095 (from `main` `n-1`, line 

## 6단계: 구조화된 보고서 만들기

후속 시스템(이슈 트래커, 대시보드, SIEM)에는 구조화된 데이터가 필요합니다. 마지막으로 도구 없는 `query()`가 선별된 발견 사항을 명시적 스키마를 따르는 JSON으로 변환합니다. 모델이 선택 여부를 추측하지 않도록 모든 필드를 필수로 표시하고 "해당 없음"에는 `null`을 씁니다. 프로덕션에서는 `jsonschema`로 검증하고 실패 시 재시도하게 됩니다.

In [7]:
# Every key is in `required` so the model never silently drops a field; values
# may be null when a field is not applicable (e.g., no recommendation yet).
REPORT_SCHEMA = {
    "type": "object",
    "properties": {
        "findings": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": ["string", "null"]},
                    "category": {"type": ["string", "null"]},
                    "severity": {"type": "string", "enum": ["critical", "high", "medium", "low"]},
                    "file": {"type": ["string", "null"]},
                    "description": {"type": ["string", "null"]},
                    "recommendation": {"type": ["string", "null"]},
                },
                "required": ["id", "category", "severity", "file", "description", "recommendation"],
            },
        }
    },
    "required": ["findings"],
}

REPORT_PROMPT = f"""\
Convert the triaged findings below into strict JSON conforming to this schema.
Every field is required; use null for not-applicable. Respond with JSON only,
no surrounding prose or code fences.

## Schema

{json.dumps(REPORT_SCHEMA, indent=2)}

## Triaged findings

{triaged_text}
"""

report_options = ClaudeAgentOptions(
    model=MODEL_NAME,
    system_prompt={"type": "preset", "preset": "claude_code", "append": ENGAGEMENT_CONTEXT},
    allowed_tools=[],
)

report_json = await collect(query(prompt=REPORT_PROMPT, options=report_options))
raw = report_json.strip()
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
try:
    report = json.loads(raw)
except json.JSONDecodeError as e:
    print(f"[report agent did not return clean JSON: {e}]\n")
    print(raw)
else:
    print(json.dumps(report, indent=2))

{
  "findings": [
    {
      "id": "F-01",
      "category": "Heap buffer overflow",
      "severity": "critical",
      "file": "canary.c",
      "description": "parse_alpha allocates malloc(32) then memcpys attacker-controlled data of length up to 4095 bytes (from main's n-1 at line 39). An attacker-authored file beginning with 'A' followed by >=33 bytes overflows the heap allocation. With no sandbox, this yields RCE as the invoking user.",
      "recommendation": "Validate len against the allocation size before memcpy (e.g., require len <= 32) or size the allocation from len. Reject oversized inputs at the parser boundary and add bounds-checked copy helpers."
    },
    {
      "id": "F-02",
      "category": "Stack buffer overflow",
      "severity": "critical",
      "file": "canary.c",
      "description": "parse_bravo declares char name[16] then memcpys attacker-controlled data of length up to 4095 bytes into it. The name[15]=0 null-termination on line 17 executes after the out

## 요약과 다음 단계

Agent SDK로 위협 모델링, 탐색, 선별, 보고까지 전체 파이프라인을 만들었습니다. 위협 모델에는 멀티턴 `ClaudeSDKClient` 세션 하나를, 나머지에는 일회성 `query()` 세 번을 사용했고, 탐색은 Claude Code의 내장 파일 도구가 담당했습니다. 가져갈 핵심 패턴은 다음과 같습니다.

- **대화에는 `ClaudeSDKClient`, 일회성에는 `query()`.** 위협 모델 인터뷰는 부트스트랩 턴이 컨텍스트에 있어야 하지만, 탐색·선별·보고는 서로의 도구 기록에 의존하지 않으므로 상태 없는 호출이 더 단순합니다.
- **`cwd` + `allowed_tools`가 직접 만든 도구를 대체합니다.** 대상 디렉터리로 범위를 좁힌 `Read`/`Grep`/`Glob`이 탐색 에이전트 뼈대의 전부입니다.
- **선별은 별도의 단계입니다.** 탐색 에이전트와 독립적으로 다시 검증하고 다시 점수를 매기는 것이 과신을 값싸게 잡아냅니다.

### 한 걸음 더

- **호스팅 버전 사용하기.** [Claude Code Security](https://claude.com/claude-code-security)는 동일한 탐색·선별 기능을 관리형 제품으로 제공합니다. 저장소를 가리키기만 하면 샌드박싱과 확장은 Anthropic이 처리합니다.
- **실제 저장소로 확장하기.** `cwd`를 실제 체크아웃으로 가리키고, 샌드박스 컨테이너 안에서 `allowed_tools`에 `"Bash"`를 추가해 에이전트가 `-fsanitize=address`로 컴파일해 크래시를 확인하게 하고, 위협 모델의 진입점마다 `asyncio.gather`로 `query()`를 하나씩 띄우세요.
- **보고서를 트래커에 연결하기.** 6단계의 JSON을 `jsonschema`로 검증하고, SARIF나 여러분의 티켓 스키마로 매핑해 POST하세요.